# Linear Regression — Solution

**Dataset:** Rental apartments in Zurich: area, rooms and monthly rent.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import statsmodels.api as sm
import statsmodels.formula.api as smf

## 1. Calculate OLS with three apartments

The OLS formula is in the concept notebook. Replace the gap with `np.sum((x - x_mean) * (y - y_mean))`. The remaining calculation is provided. What are the slope and intercept? Is the 95 m² prediction interpolation or extrapolation for these three flats?

In [ ]:
x = np.array([40, 60, 80])
y = np.array([1200, 1600, 2000])
x_mean = x.mean()
y_mean = y.mean()
numerator = np.sum((x - x_mean) * (y - y_mean))
denominator = np.sum((x - x_mean) ** 2)
slope = numerator / denominator
intercept = y_mean - slope * x_mean
print('Slope:', slope, 'Intercept:', intercept)
print('Predicted rent for 95 m²:', intercept + slope * 95)
# A separate R² illustration from slide 22:
print('R² for SSE=20 and SST=100:', 1 - 20 / 100)

**Answer:** Slope = 20 and intercept = 400. The prediction is 2300 CHF/month, beyond the observed 40–80 m² range: extrapolation. One extra m² corresponds to 20 CHF more predicted rent. In the separate R² example, 0.8 means 80% of the training target variation is explained.

## 2. Load apartments

Preparation is provided. What does a point in this plot represent? Describe the overall pattern.

In [ ]:
apt = pd.read_csv('../00_Data/rental_apartments_canton_zh.csv', sep=';')
apt = apt[['bfs_name', 'rooms', 'area', 'price']].dropna().drop_duplicates()
apt = apt[(apt['price'] >= 1000) & (apt['price'] <= 5000)]
train, test = train_test_split(apt, test_size=0.2, random_state=42)
print(train[['area', 'rooms', 'price']].describe().round(1))
plt.scatter(train['area'], train['price'], alpha=0.4)
plt.xlabel('Living area [m²]')
plt.ylabel('Rent [CHF/month]')
plt.show()

**Answer:** Each point is an apartment. Larger flats generally have higher rents, but flats of similar area still differ in rent.

## 3. Fit simple OLS

Replace the gap with `smf.ols("price ~ area", data=train).fit()`. Interpret the printed area coefficient.

In [ ]:
simple = smf.ols("price ~ area", data=train).fit()
print(simple.params.round(2))
print('Training R²:', round(simple.rsquared, 3))
grid = pd.DataFrame({'area': np.linspace(train['area'].min(), train['area'].max(), 100)})
plt.scatter(train['area'], train['price'], alpha=0.3)
plt.plot(grid['area'], simple.predict(grid), color='red')
plt.xlabel('Living area [m²]')
plt.ylabel('Rent [CHF/month]')
plt.show()

**Answer:** The area coefficient gives the estimated CHF/month increase per extra m². The intercept concerns zero area and is not a realistic apartment rent.

## 4. Read intervals for 95 m²

Which columns describe average rent? Which interval concerns one new apartment?

In [ ]:
new_flat = pd.DataFrame({'area': [95]})
intervals = simple.get_prediction(new_flat).summary_frame(alpha=0.05)
print(intervals[['mean', 'mean_ci_lower', 'mean_ci_upper',
                 'obs_ci_lower', 'obs_ci_upper']].round(0))

**Answer:** The mean_ci columns bound the estimated average rent. The wider obs_ci interval concerns one new apartment.

## 5. Add rooms

Replace the gap with `smf.ols("price ~ area + rooms", data=train).fit()`. What does the area coefficient now mean?

In [ ]:
multiple = smf.ols("price ~ area + rooms", data=train).fit()
print(multiple.params.round(2))
print('Training R²:', round(multiple.rsquared, 3))
print('Input correlations:')
print(train[['area', 'rooms']].corr().round(2))

**Answer:** The coefficient describes one extra m² while keeping rooms constant. Correlated inputs make separate coefficients harder to estimate. An extra input need not improve new predictions.

## 6. Check residuals

Do you see a curve or a funnel in the residual plot? Can the Q–Q plot establish independence?

In [ ]:
plt.scatter(simple.fittedvalues, simple.resid, alpha=0.4)
plt.axhline(0, color='black')
plt.xlabel('Fitted rent [CHF]')
plt.ylabel('Residual [CHF]')
plt.show()
sm.qqplot(simple.resid, line='s')
plt.title('Normal Q–Q plot')
plt.show()

**Answer:** A curve suggests missing structure; a funnel suggests changing variance. Q–Q departures suggest non-normal errors. Independence must be assessed from the study design.

## 7. Final test evaluation

These two models are fixed before this test comparison. Which has lower RMSE? What does RMSE of 500 CHF mean? Use CV for further model choices, rather than repeated test comparisons.

In [ ]:
simple_pred = simple.predict(test)
multiple_pred = multiple.predict(test)
print('Simple test RMSE:', round(np.sqrt(mean_squared_error(test['price'], simple_pred)), 1))
print('Multiple test RMSE:', round(np.sqrt(mean_squared_error(test['price'], multiple_pred)), 1))
print('Simple test R²:', round(r2_score(test['price'], simple_pred), 3))
print('Multiple test R²:', round(r2_score(test['price'], multiple_pred), 3))

**Answer:** Compare the printed errors. RMSE of 500 is a root mean squared error of 500 CHF/month, not an error bound for each flat. The extra input may or may not improve test performance.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))